<a href="https://colab.research.google.com/github/Paulo-Vitor-Tirol-Gama/excel-copilot-attendance-automation/blob/main/excel_copilot_attendance_automatio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nova seção

In [ ]:
!pip install pandas openpyxl


In [ ]:
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.formatting.rule import FormulaRule, DataBarRule
from openpyxl.utils import get_column_letter
from datetime import date

# Definição limpa dos caminhos físicos dos arquivos (Sem caracteres especiais)
OUTPUT_FINAL = "Attendance_Tracker_Final.xlsx"
OUTPUT_TEMPLATE = "Attendance_Tracker_Template.xlsx"

# Massa de dados exata extraída do log enviado pelo aluno
list1_data = [
    ("Ana Silva", "ana.silva@example.com"), ("Bruno Costa", "bruno.costa@example.com"),
    ("Carla Souza", "carla.souza@example.com"), ("Daniel Lima", "daniel.lima@example.com"),
    ("Eduarda Alves", "eduarda.alves@example.com"), ("Felipe Santos", "felipe.santos@example.com"),
    ("Gabriela Rocha", "gabriela.rocha@example.com"), ("Henrique Martins", "henrique.martins@example.com"),
    ("Isabela Ferreira", "isabela.ferreira@example.com"), ("Joao Oliveira", "joao.oliveira@example.com"),
    ("Larissa Mendes", "larissa.mendes@example.com"), ("Marcos Ribeiro", "marcos.ribeiro@example.com")
]

list2_data = [
    ("Ana Silva", "A"), ("Bruno Costa", "B"), ("Carla Souza", "C"), ("Daniel Lima", "D"),
    ("Eduarda Alves", "E"), ("Felipe Santos", "A"), ("Gabriela Rocha", "B"), ("Henrique Martins", "C"),
    ("Isabela Ferreira", "D"), ("Joao Oliveira", "E"), ("Larissa Mendes", "A"), ("Marcos Ribeiro", "B")
]

list3_data = [
    ("Ana Silva","Monday","2024-07-01","Yes"), ("Bruno Costa","Monday","2024-07-01","Yes"),
    ("Carla Souza","Monday","2024-07-01","No"), ("Daniel Lima","Monday","2024-07-01","Yes"),
    ("Eduarda Alves","Monday","2024-07-01","Yes"), ("Felipe Santos","Monday","2024-07-01","Yes"),
    ("Gabriela Rocha","Monday","2024-07-01","Yes"), ("Henrique Martins","Monday","2024-07-01","Yes"),
    ("Isabela Ferreira","Monday","2024-07-01","No"), ("Joao Oliveira","Monday","2024-07-01","Yes"),
    ("Larissa Mendes","Monday","2024-07-01","Yes"), ("Marcos Ribeiro","Monday","2024-07-01","Yes"),
    ("Ana Silva","Tuesday","2024-07-02","Yes"), ("Bruno Costa","Tuesday","2024-07-02","Yes"),
    ("Carla Souza","Tuesday","2024-07-02","Yes"), ("Daniel Lima","Tuesday","2024-07-02","Yes"),
    ("Eduarda Alves","Tuesday","2024-07-02","No"), ("Felipe Santos","Tuesday","2024-07-02","Yes"),
    ("Gabriela Rocha","Tuesday","2024-07-02","No"), ("Henrique Martins","Tuesday","2024-07-02","No"),
    ("Isabela Ferreira","Tuesday","2024-07-02","Yes"), ("Joao Oliveira","Tuesday","2024-07-02","Yes"),
    ("Larissa Mendes","Tuesday","2024-07-02","No"), ("Marcos Ribeiro","Tuesday","2024-07-02","No"),
    ("Ana Silva","Wednesday","2024-07-03","No"), ("Bruno Costa","Wednesday","2024-07-03","Yes"),
    ("Carla Souza","Wednesday","2024-07-03","Yes"), ("Daniel Lima","Wednesday","2024-07-03","No"),
    ("Eduarda Alves","Wednesday","2024-07-03","Yes"), ("Felipe Santos","Wednesday","2024-07-03","Yes"),
    ("Gabriela Rocha","Wednesday","2024-07-03","No"), ("Henrique Martins","Wednesday","2024-07-03","Yes"),
    ("Isabela Ferreira","Wednesday","2024-07-03","No"), ("Joao Oliveira","Wednesday","2024-07-03","No"),
    ("Larissa Mendes","Wednesday","2024-07-03","No"), ("Marcos Ribeiro","Wednesday","2024-07-03","Yes"),
    ("Ana Silva","Thursday","2024-07-04","No"), ("Bruno Costa","Thursday","2024-07-04","Yes"),
    ("Carla Souza","Thursday","2024-07-04","No"), ("Daniel Lima","Thursday","2024-07-04","Yes"),
    ("Eduarda Alves","Thursday","2024-07-04","No"), ("Felipe Santos","Thursday","2024-07-04","No"),
    ("Gabriela Rocha","Thursday","2024-07-04","Yes"), ("Henrique Martins","Thursday","2024-07-04","Yes"),
    ("Isabela Ferreira","Thursday","2024-07-04","Yes"), ("Joao Oliveira","Thursday","2024-07-04","Yes"),
    ("Larissa Mendes","Thursday","2024-07-04","No"), ("Marcos Ribeiro","Thursday","2024-07-04","Yes"),
    ("Ana Silva","Friday","2024-07-05","Yes"), ("Bruno Costa","Friday","2024-07-05","Yes"),
    ("Carla Souza","Friday","2024-07-05","No"), ("Daniel Lima","Friday","2024-07-05","No"),
    ("Eduarda Alves","Friday","2024-07-05","No"), ("Felipe Santos","Friday","2024-07-05","No"),
    ("Gabriela Rocha","Friday","2024-07-05","Yes"), ("Henrique Martins","Friday","2024-07-05","No"),
    ("Isabela Ferreira","Friday","2024-07-05","No"), ("Joao Oliveira","Friday","2024-07-05","Yes"),
    ("Larissa Mendes","Friday","2024-07-05","No"), ("Marcos Ribeiro","Friday","2024-07-05","Yes")
]

headers = [
    "Name", "Month", "Contact", "Group/Department",
    "Total days present", "Actual days present",
    "Attendance percentage", "Attendance requirement met"
]

# Estilos Visuais Corporativos
HEADER_FILL = PatternFill(fill_type="solid", fgColor="D9EAF7")
HEADER_FONT = Font(name="Calibri", size=11, bold=True, color="17365D")
BODY_FONT = Font(name="Calibri", size=10, color="333333")
GREEN_FILL = PatternFill(fill_type="solid", fgColor="C6EFCE")
RED_FILL = PatternFill(fill_type="solid", fgColor="FFC7CE")
THIN_BORDER = Border(bottom=Side(style="thin", color="D9E1F2"))

def build_base_sheets(wb):
    ws_l1 = wb.create_sheet("List1")
    ws_l2 = wb.create_sheet("List2")
    ws_l3 = wb.create_sheet("List3")

    ws_l1.append(["Name", "Contact"])
    for row in list1_data: ws_l1.append(row)

    ws_l2.append(["Name", "Group/Department"])
    for row in list2_data: ws_l2.append(row)

    ws_l3.append(["Name", "Day of Week", "Date", "Attended"])
    for name, day, dt_str, att in list3_data:
        y, m, d = map(int, dt_str.split("-"))
        ws_l3.append([name, day, date(y, m, d), att])

    for ws in [ws_l1, ws_l2, ws_l3]:
        ws.freeze_panes = "A2"
        ws.row_dimensions[1].height = 25
        for cell in ws[1]:
            cell.fill = HEADER_FILL
            cell.font = HEADER_FONT
            cell.alignment = Alignment(horizontal="center", vertical="center")
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.font = BODY_FONT
                cell.alignment = Alignment(horizontal="center", vertical="center")
                cell.border = THIN_BORDER
    for r in range(2, ws_l3.max_row + 1):
        ws_l3.cell(row=r, column=3).number_format = "dd/mm/yyyy"

    ws_l1.column_dimensions["A"].width = 25
    ws_l1.column_dimensions["B"].width = 35
    ws_l2.column_dimensions["A"].width = 25
    ws_l2.column_dimensions["B"].width = 20
    ws_l3.column_dimensions["A"].width = 25
    ws_l3.column_dimensions["B"].width = 15
    ws_l3.column_dimensions["C"].width = 15
    ws_l3.column_dimensions["D"].width = 12

# 1. GERAÇÃO DO ARQUIVO FINAL CALCULADO
wb_final = Workbook()
wb_final.remove(wb_final.active)
build_base_sheets(wb_final)
ws_tr1 = wb_final.create_sheet("Attendance tracker")
ws_tr1.append(headers)

for index, (name, _) in enumerate(list1_data, start=2):
    ws_tr1.cell(row=index, column=1, value=name)
    ws_tr1.cell(row=index, column=2, value=date(2024, 7, 1))
    ws_tr1.cell(row=index, column=3, value=f'=XLOOKUP(A{index},List1!A:A,List1!B:B,"Não Encontrado")')
    ws_tr1.cell(row=index, column=4, value=f'=XLOOKUP(A{index},List2!A:A,List2!B:B,"Sem Departamento")')
    ws_tr1.cell(row=index, column=5, value='=DATE(2024,7,5)-DATE(2024,7,1)+1')
    ws_tr1.cell(row=index, column=6, value=f'=COUNTIFS(List3!A:A,A{index},List3!D:D,"Yes")')
    ws_tr1.cell(row=index, column=7, value=f'=IFERROR(F{index}/E{index},0)')
    ws_tr1.cell(row=index, column=8, value=f'=IF(OR(D{index}="A",D{index}="B",D{index}="C"),IF(G{index}>=0.8,"Met","Not Met"),IF(G{index}>=0.4,"Met","Not Met"))')

# Formatação do Tracker Final
ws_tr1.freeze_panes = "A2"
ws_tr1.row_dimensions[1].height = 25
for cell in ws_tr1[1]:
    cell.fill = HEADER_FILL
    cell.font = HEADER_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
for row in ws_tr1.iter_rows(min_row=2):
    for cell in row:
        cell.font = BODY_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = THIN_BORDER
    row[1].number_format = "mmm-yyyy"
    row[6].number_format = "0.0%"

mx_r = ws_tr1.max_row
mx_c = get_column_letter(ws_tr1.max_column)
t_obj1 = Table(displayName="AttendanceTableFinal", ref=f"A1:{mx_c}{mx_r}")
t_obj1.tableStyleInfo = TableStyleInfo(name="TableStyleMedium9", showRowStripes=True)
ws_tr1.add_table(t_obj1)
ws_tr1.conditional_formatting.add(f"H2:H{mx_r}", FormulaRule(formula=['H2="Met"'], fill=GREEN_FILL))
ws_tracker.conditional_formatting.add(f"H2:H{mx_r}", FormulaRule(formula=['H2="Not Met"'], fill=RED_FILL))
ws_tr1.conditional_formatting.add(f"G2:G{mx_r}", DataBarRule(start_type="num", start_value=0, end_type="num", end_value=1, color="FF5B9BD5", showValue="None"))

for col in ws_tr1.columns:
    ws_tr1.column_dimensions[get_column_letter(col[0].column)].width = 22
wb_final.save(OUTPUT_FINAL)

# 2. GERAÇÃO DO TEMPLATE LIMPO (ESQUELETO)
wb_temp = Workbook()
wb_temp.remove(wb_temp.active)
build_base_sheets(wb_temp)
ws_tr2 = wb_temp.create_sheet("Attendance tracker")
ws_tr2.append(headers)

for index, (name, _) in enumerate(list1_data, start=2):
    ws_tr2.cell(row=index, column=1, value=name)
    ws_tr2.cell(row=index, column=2, value=date(2024, 7, 1))

ws_tr2.freeze_panes = "A2"
ws_tr2.row_dimensions[1].height = 25
for cell in ws_tr2[1]:
    cell.fill = HEADER_FILL
    cell.font = HEADER_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
for row in ws_tr2.iter_rows(min_row=2):
    for cell in row:
        cell.font = BODY_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = THIN_BORDER
    row[1].number_format = "mmm-yyyy"

t_obj2 = Table(displayName="AttendanceTableTemplate", ref=f"A1:{mx_c}{mx_r}")
t_obj2.tableStyleInfo = TableStyleInfo(name="TableStyleMedium9", showRowStripes=True)
ws_tr2.add_table(t_obj2)

for col in ws_tr2.columns:
    ws_tr2.column_dimensions[get_column_letter(col[0].column)].width = 22


NameError: name 'ws_tracker' is not defined